# 05 · AutoML en handmatige modellen

**Auteurs:** _vul namen en concrete bijdragen in vóór indiening_  
**Doel:** Een baseline, AutoML en meerdere uitlegbare kandidaten eerlijk vergelijken.

Elke codecel wordt voorafgegaan door uitleg. Voer de notebooks in nummervolgorde uit
vanaf de repository-root. Resultaten moeten door het team zelf worden gecontroleerd en
geïnterpreteerd; synthetische CI-data gelden nooit als onderzoeksbewijs.

## Evaluatieontwerp

De laatste 20% van ritten vormt de onaangeraakte hold-outset. Een random split zou
toekomstige stations- en gedragspatronen naar training lekken. MAE is primair; RMSE,
MedAE en R² maken verschillende foutaspecten zichtbaar. Features bevatten uitsluitend
informatie die bij vertrek bekend is.

In [ ]:
from src.config import PROCESSED_DIR, MODEL_DIR, REPORT_DIR
from src.models.train import train_and_compare

bundle, metrics = train_and_compare(
    PROCESSED_DIR / "trips.parquet",
    MODEL_DIR / "duration_model.joblib",
    REPORT_DIR / "metrics.csv",
    max_rows=500_000,
    quick=False,
)
display(metrics.style.format({"mae_minutes":"{:.2f}", "rmse_minutes":"{:.2f}", "r2":"{:.3f}"}))

## Waarom deze modellen?

De mediaan is de eerlijke naïeve referentie. Ridge test een eenvoudige lineaire relatie.
Histogram Gradient Boosting vangt niet-lineaire interacties efficiënt. Extra Trees kan
complexere stations- en tijdspatronen leren, maar overfit makkelijker. De target krijgt
een log1p-transformatie wegens de lange rechterstaart.

## AutoML (FLAML)

FLAML vervangt hier PyCaret als toegestane AutoML-vergelijking. Het tijdsbudget maakt de run reproduceerbaar qua kosten; de uitkomst kan per hardware iets verschillen.

In [ ]:
RUN_AUTOML = False
if RUN_AUTOML:
    from src.models.automl import run_automl
    automl_result = run_automl(PROCESSED_DIR / "trips.parquet", seconds=600)
    display(automl_result)
else:
    print("Zet RUN_AUTOML=True na installatie van de optionele FLAML dependency.")

## Selectie

De pipeline kiest de laagste hold-out-MAE, niet de trainingsscore. Controleer in notebook 06 of winst breed gedragen wordt en niet uit één subgroup komt.